# 01 — Cohort Audit

Loads `data/processed/cohort_features.parquet` (the real MIMIC-IV Clinical Database Demo cohort built by `src/build_demo_cohort.py`), reports cohort size/exclusions, mortality prevalence, missingness, and summary statistics; performs a patient-level 70/10/20 stratified split; fits preprocessing on TRAIN only; saves the frozen preprocessor.

**All numbers below are computed directly from the real cohort file — nothing is simulated or fabricated.**

In [ ]:
import sys
sys.path.append('..')

from pathlib import Path

import joblib
import pandas as pd

from src.config import load_config
from src.reproducibility import set_global_seed, get_logger, log_run_metadata
from src.preprocess import (
    load_cohort_features,
    check_duplicates,
    drop_duplicate_rows,
    patient_level_split,
    check_patient_overlap,
    class_distribution,
    drop_identifier_columns,
    identify_feature_types,
    ClinicalPreprocessor,
    generate_table1,
    IDENTIFIER_COLUMNS,
)

config = load_config()
seed = config["project"]["random_seed"]
set_global_seed(seed)
logger = get_logger(log_file=config["logging"]["log_file"])
logger.info("01_cohort_audit started (seed=%d)", seed)

ID_COL = config["preprocessing"]["id_column"]
TARGET_COL = config["preprocessing"]["target_column"]
TABLES_DIR = Path(config["training"]["tables_output_dir"])
TABLES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)

## 1. Load cohort, report total ICU stays, exclusions, final N

Raises `FileNotFoundError` with a clear message if the real cohort file has not been built yet (via `python -m src.build_demo_cohort`) — no placeholder or synthetic data is used.

In [ ]:
raw_df = load_cohort_features(config=config)
n_loaded = len(raw_df)
print(f"Total ICU stays loaded: {n_loaded}")

dup_result = check_duplicates(raw_df, id_col=ID_COL)
print(f"Fully duplicate rows: {dup_result.n_fully_duplicate_rows}")
print(f"Duplicate {ID_COL} values (>1 row per patient): {dup_result.n_duplicate_ids}")

dedup_df = drop_duplicate_rows(raw_df)
n_excluded = n_loaded - len(dedup_df)
n_final = len(dedup_df)

print(f"\nExcluded (duplicate rows): {n_excluded}")
print(f"Final N (analysis cohort): {n_final}")

cohort_flow_df = pd.DataFrame([
    {"stage": "Total ICU stays loaded", "n": n_loaded},
    {"stage": "Excluded: duplicate rows", "n": n_excluded},
    {"stage": "Final analysis cohort (N)", "n": n_final},
])
cohort_flow_df.to_csv(TABLES_DIR / "cohort_flow_counts.csv", index=False)
cohort_flow_df

## 2. Mortality prevalence

In [ ]:
dist = class_distribution(dedup_df, target_col=TARGET_COL)
n_deaths = int(dedup_df[TARGET_COL].sum())
pct_deaths = 100 * dedup_df[TARGET_COL].mean()

print(f"N = {n_final}")
print(f"Deaths (in-hospital mortality = 1): {n_deaths} ({pct_deaths:.2f}%)")
print(f"Survivors (mortality = 0): {n_final - n_deaths} ({100 - pct_deaths:.2f}%)")
dist

## 3. Missingness per feature (count and percentage)

In [ ]:
feature_df = drop_identifier_columns(dedup_df, IDENTIFIER_COLUMNS)

missingness_df = pd.DataFrame({
    "n_missing": feature_df.isna().sum(),
    "pct_missing": (100 * feature_df.isna().mean()).round(2),
}).sort_values("pct_missing", ascending=False)
missingness_df.to_csv(TABLES_DIR / "cohort_missingness.csv")
missingness_df

## 4. Demographic and clinical summary statistics

In [ ]:
# LEAKAGE GUARD: "los" (ICU length of stay) and "outtime" are known only
# at ICU discharge, not during the first 24h, and strongly correlate with
# mortality (very short and very long stays both skew toward death) — the
# same leakage pattern originally caught and fixed in sql/cohort.sql.
# Excluded here so no downstream notebook (02+) can ever train on them,
# since they all reuse this notebook's frozen preprocessor/numeric_cols.
model_exclude_cols = [TARGET_COL, "icu_intime", "intime", "outtime", "los"]
numeric_cols, categorical_cols = identify_feature_types(feature_df, exclude_cols=model_exclude_cols)
print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

feature_df[numeric_cols].describe().T

## 5. Patient-level 70% / 10% / 20% split, stratified by mortality

In [ ]:
train_df, val_df, test_df = patient_level_split(
    dedup_df, id_col=ID_COL, target_col=TARGET_COL,
    train_size=config["preprocessing"]["train_size"],
    val_size=config["preprocessing"]["val_size"],
    test_size=config["preprocessing"]["test_size"],
    seed=seed,
)
check_patient_overlap(train_df, val_df, test_df, id_col=ID_COL)
print("Patient-overlap check passed: no patient id appears in more than one split.\n")

for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    n = len(split_df)
    deaths = int(split_df[TARGET_COL].sum())
    print(f"{name:5s}: n={n:4d} ({n/n_final:.1%}), deaths={deaths} ({deaths/n:.1%})")

## 6. Fit preprocessing on TRAIN only

Median imputation + missingness indicators for numeric features, one-hot encoding for categorical features, standard scaling for numeric features — all fit exclusively on `train_df` (`src.preprocess.ClinicalPreprocessor`). `val_df`/`test_df` are transformed using these frozen statistics, never refit.

In [ ]:
train_features_df = drop_identifier_columns(train_df, IDENTIFIER_COLUMNS)
val_features_df = drop_identifier_columns(val_df, IDENTIFIER_COLUMNS)
test_features_df = drop_identifier_columns(test_df, IDENTIFIER_COLUMNS)

numeric_cols, categorical_cols = identify_feature_types(train_features_df, exclude_cols=model_exclude_cols)

preprocessor = ClinicalPreprocessor(numeric_cols=numeric_cols, categorical_cols=categorical_cols)
preprocessor.fit(train_features_df)  # TRAIN ONLY

X_train = preprocessor.transform(train_features_df)
X_val = preprocessor.transform(val_features_df)
X_test = preprocessor.transform(test_features_df)  # transformed now, but NOT looked at until notebook 04

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")
print(f"Missingness indicator columns added: {preprocessor.missing_indicator_cols_}")

preprocessor_path = config["preprocessing"]["preprocessor_output_path"]
Path(preprocessor_path).parent.mkdir(parents=True, exist_ok=True)
joblib.dump(preprocessor, preprocessor_path)
logger.info("Fitted preprocessor saved to %s", preprocessor_path)
print(f"\nSaved fitted preprocessor to {preprocessor_path}")

## 7. Table 1: cohort characteristics, overall and by split

In [ ]:
labeled_df = pd.concat([
    train_df.assign(split="train"),
    val_df.assign(split="val"),
    test_df.assign(split="test"),
], ignore_index=True)
labeled_feature_df = drop_identifier_columns(labeled_df, IDENTIFIER_COLUMNS)

table1_df = generate_table1(
    labeled_feature_df, numeric_cols=numeric_cols, categorical_cols=categorical_cols, group_col="split",
)

table1_csv_path = TABLES_DIR / "table_1_cohort.csv"
table1_df.to_csv(table1_csv_path)

table1_md_path = TABLES_DIR / "table_1_cohort.md"
md_text = (
    "**Table 1. Cohort characteristics, overall and by data split.**\n\n"
    + table1_df.reset_index().to_markdown(index=False)
    + "\n\n*Numeric variables: median [interquartile range]. Categorical variables: n (%).*\n"
    + f"\n*N = {n_final}; deaths = {n_deaths} ({pct_deaths:.1f}%); splits are patient-level (no overlap).*\n"
)
table1_md_path.write_text(md_text, encoding="utf-8")

logger.info("Table 1 saved to %s and %s", table1_csv_path, table1_md_path)
print(f"Saved {table1_csv_path} and {table1_md_path}")
table1_df

## 8. Run metadata

In [ ]:
log_run_metadata(
    seed=seed,
    extra={
        "notebook": "01_cohort_audit",
        "n_loaded": n_loaded,
        "n_excluded_duplicates": n_excluded,
        "n_final": n_final,
        "n_deaths": n_deaths,
        "mortality_prevalence_pct": round(pct_deaths, 2),
        "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
        "numeric_features": numeric_cols,
        "categorical_features": categorical_cols,
        "test_set_touched": False,
    },
)
logger.info("01_cohort_audit finished")
print(f"\nSummary: N={n_final}, deaths={n_deaths} ({pct_deaths:.1f}%), "
      f"train/val/test = {len(train_df)}/{len(val_df)}/{len(test_df)}")